# Подготовка

## Импорты

In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

from src.dataset import DatasetStore
from src.runner import ExperimentRunner
from src.pipeline import Pipeline
from src.preprocessors import *
from src.models import *
from src.validators import *

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Загрузка данных

In [3]:
df, schema = DatasetStore.load(
    "data/processed/dataset.csv",
    "data/processed/dataset.schema.json",
    include_missing=False,
)
feature_info = DatasetStore.get_feature_info(
    schema,
    include_missing=False,
)

In [4]:
columns = ['bled cut', 'albumin']
for col in columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [5]:
df["diagnosis"] = df["diagnosis"].replace(6, 5)

y = LabelEncoder().fit_transform(df['diagnosis'].values)
X = df.drop(columns=['diagnosis'])

class_labels = [0, 1]

## Инициализация компонентов

In [6]:
onehot_prep = OneHotPreprocessor(
    categorical_features=feature_info['categorical_features'],
    ordinal_features=feature_info['ordinal_features'],
    ordinal_categories=feature_info['ordinal_categories'],
    missing_value=feature_info['missing_value'],
)

In [7]:
validator = RepeatedStratifiedKFoldValidator(
    n_splits=3, n_repeats=10, random_state=42,
)

In [8]:
runner = ExperimentRunner(results_dir='_results')

# Обучение

In [8]:
logreg_base = LogisticRegressionModel(
    C=0.5,
    solver="saga",
    class_weight="balanced",
    max_iter=100,
)

In [ ]:
runner.run_single(
    name="LogReg",
    preprocessor=onehot_prep,
    model=logreg_base,
    validator=validator,
    X=X, y=y,
    class_labels=class_labels,
    save_pipeline=True,
)